# Experiment 006-B — ACT学習（最終実験）

**Experiment 005と同じGoogle Colab環境構築手順**を使用します。Python 3.12専用の`uv`仮想環境、RMBの固定コミット `8535948420f8f9bc979fdb31ed049d1c82b906b2`、ACT（Batch 4 / Chunk 25 / Skip 2 / Seed 42）を維持します。

**今回のみの変更点**：新規HDF5 100件、固定層化分割（Train80/Val20）、100エポック、正規化統計量は学習80件のみ、保存先 `act_experiment_006b`。既存005/006-Aのモデルは上書きしません。ACT重みは新規初期化します。

**準備**：Google Driveの `MyDrive/02_projects/manipulation_lab/data/rmb/pick_and_place_v006b/` に `episode_0001.hdf5`〜`episode_0100.hdf5` をアップロードし、`training/experiment_006b_split.json` に固定分割JSONをアップロードしてください。データとJSONはWSL2で生成済みです。

セルを**上から順に**実行します。データのチェックが通るまで学習は始まりません。

## 1. Google Driveを接続・100件の配置を確認

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/02_projects/manipulation_lab')
DATASET_DIR = DRIVE_ROOT / 'data/rmb/pick_and_place_v006b'
SPLIT_JSON = DRIVE_ROOT / 'training/experiment_006b_split.json'
CHECKPOINT_DIR = DRIVE_ROOT / 'training/checkpoints/act_experiment_006b'

assert DRIVE_ROOT.is_dir(), f'Drive root not found: {DRIVE_ROOT}'
files = sorted(DATASET_DIR.glob('episode_*.hdf5'))
print('Dataset:', DATASET_DIR)
print('HDF5:', len(files))
print('Split JSON:', SPLIT_JSON.is_file())
assert len(files) == 100, 'episode_0001〜episode_0100.hdf5を100件アップロードしてください'
assert SPLIT_JSON.is_file(), f'Split JSON is missing: {SPLIT_JSON}'
assert not (CHECKPOINT_DIR.exists() and any(CHECKPOINT_DIR.iterdir())), 'Checkpoint保存先が空ではありません。上書きを防ぐため停止します。'


## 2. GPU確認（005と同じ）

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], text=True, capture_output=True)
print(result.stdout or result.stderr)
if result.returncode != 0:
    raise RuntimeError('GPUランタイムを有効にしてください')

## 3. Python 3.12 / RoboManipBaselines / ACT環境構築（005と同じセル）

In [ ]:
import subprocess
import sys
from pathlib import Path

RMB_REPO = Path('/content/RoboManipBaselines')
SHA = '8535948420f8f9bc979fdb31ed049d1c82b906b2'
VENV_DIR = Path('/content/act-venv')
PYTHON312 = VENV_DIR / 'bin/python'

def run(*command, cwd=None):
    command = [str(x) for x in command]
    print('+', ' '.join(command), flush=True)
    subprocess.run(command, cwd=cwd, check=True)

run(sys.executable, '-m', 'pip', 'install', '-q', 'uv')
run(sys.executable, '-m', 'uv', 'python', 'install', '3.12')
if not PYTHON312.exists():
    run(sys.executable, '-m', 'uv', 'venv', '--python', '3.12', str(VENV_DIR))
run(PYTHON312, '--version')
assert subprocess.check_output([str(PYTHON312), '-c', 'import sys; print(sys.version_info[:2])'], text=True).strip() == '(3, 12)'

if not (RMB_REPO / '.git').exists():
    run('git', 'clone', 'https://github.com/isri-aist/RoboManipBaselines.git', RMB_REPO)
run('git', 'checkout', SHA, cwd=RMB_REPO)
run('git', 'submodule', 'update', '--init', 'third_party/act', cwd=RMB_REPO)

run(sys.executable, '-m', 'uv', 'pip', 'install', '--python', PYTHON312, '-e', '.[act]', cwd=RMB_REPO)
run(sys.executable, '-m', 'uv', 'pip', 'install', '--python', PYTHON312, '-e', '.', cwd=RMB_REPO / 'third_party/act/detr')
print('RMB/ACT install complete under Python 3.12')


## 4. データ100件と固定80/20分割の品質チェック

In [ ]:
import json
import subprocess

manifest = json.loads(SPLIT_JSON.read_text(encoding='utf-8'))
train_names = list(manifest['train'])
val_names = list(manifest['validation'])
expected = {f'episode_{i:04d}.hdf5' for i in range(1, 101)}
assert len(train_names) == 80 and len(val_names) == 20
assert set(train_names).isdisjoint(val_names)
assert set(train_names + val_names) == expected
assert {f.name for f in files} == expected

# 各分類の学習/検証数を確認
ranges = {'A': (1, 40, 32, 8), 'B': (41, 70, 24, 6),
          'C': (71, 90, 16, 4), 'D': (91, 100, 8, 2)}
for name, (start, end, ntr, nval) in ranges.items():
    subset = {f'episode_{i:04d}.hdf5' for i in range(start, end + 1)}
    assert len(set(train_names) & subset) == ntr, name
    assert len(set(val_names) & subset) == nval, name
    print(f'{name}: Train={ntr}, Validation={nval}')

validation_code = r"""
import sys, h5py, numpy as np, torch
from pathlib import Path
assert sys.version_info[:2] == (3, 12), sys.version
assert torch.cuda.is_available(), 'CUDAが利用できません'
print('Python:', sys.version.split()[0], 'Torch:', torch.__version__, 'GPU:', torch.cuda.get_device_name(0))
root = Path(sys.argv[1]); paths = sorted(root.glob('episode_*.hdf5'))
assert len(paths) == 100
for path in paths:
    with h5py.File(path, 'r') as h:
        n = len(h['time'])
        assert n == 1350, (path, n)
        assert h['measured_joint_pos'].shape == (n, 7), path
        assert h['command_joint_pos'].shape == (n, 7), path
        assert h['front_rgb_image'].shape == (n, 240, 320, 3), path
        assert h['front_rgb_image'].dtype == np.uint8, path
        assert np.all(np.diff(h['time'][:]) > 0), path
        a = h['command_joint_pos'][:]
        x = h['measured_joint_pos'][:]
        assert np.isfinite(a).all() and np.isfinite(x).all(), path
        assert ((0 <= a[:, 6]) & (a[:, 6] <= 255)).all(), path
print('HDF5 check: 100/100 OK')
"""
subprocess.run([str(PYTHON312), '-c', validation_code, str(DATASET_DIR)], check=True)
print('Train=80 Val=20 no overlap; ready for training')


## 5. 学習スクリプト（内蔵）を用意

005/006-A同様、RMB内部のファイル順序を固定する小さなラッパーを利用します。**正規化統計量は学習80件のみで計算**し、検証20件の情報を学習へ混ぜないようにします。学習コマンドは従来と同じRMB `Train.py Act` です。ソースを別途アップロードする必要はありません。


In [ ]:
from pathlib import Path
import py_compile

TRAIN_SCRIPT = Path('/content/train_act_v006b.py')
TRAIN_SCRIPT.write_text('#!/usr/bin/env python3\n"""Experiment 006-B: 100 new HDF5, fixed stratified 80/20 split, ACT from scratch."""\nfrom __future__ import annotations\nimport argparse\nimport json\nimport os\nimport runpy\nimport sys\nfrom pathlib import Path\n\n\ndef main():\n    p = argparse.ArgumentParser()\n    p.add_argument("--dataset-dir", type=Path, required=True)\n    p.add_argument("--split-json", type=Path, required=True)\n    p.add_argument("--checkpoint-dir", type=Path, required=True)\n    p.add_argument("--rmb-root", type=Path, required=True)\n    p.add_argument("--epochs", type=int, default=100)\n    p.add_argument("--batch-size", type=int, default=4)\n    p.add_argument("--chunk-size", type=int, default=25)\n    p.add_argument("--skip", type=int, default=2)\n    p.add_argument("--num-workers", type=int, default=2)\n    args = p.parse_args()\n    if sys.version_info[:2] != (3, 12):\n        raise RuntimeError(\'Python 3.12 is required\')\n    import torch\n    if not torch.cuda.is_available():\n        raise RuntimeError(\'CUDA GPU is required\')\n\n    dataset = args.dataset_dir.resolve(strict=True)\n    manifest = json.loads(args.split_json.read_text(encoding="utf-8"))\n    train = list(manifest["train"])\n    validation = list(manifest["validation"])\n    if len(train) != 80 or len(validation) != 20:\n        raise ValueError(\'Expected 80 train and 20 validation episodes\')\n    if len(set(train + validation)) != 100 or (set(train) & set(validation)):\n        raise ValueError(\'Overlapping or duplicate episode names\')\n    all_hdf5 = {f.name for f in dataset.glob(\'*.hdf5\')}\n    if all_hdf5 != set(train + validation):\n        raise ValueError(\'Dataset must contain exactly the 100 split files\')\n\n    ordered_files = [str((dataset / n).resolve(strict=True)) for n in train + validation]\n    rmb_root = args.rmb_root.resolve(strict=True)\n    entry = rmb_root / \'bin\' / \'Train.py\'\n    if not entry.is_file():\n        raise FileNotFoundError(entry)\n    checkpoint = args.checkpoint_dir.resolve()\n    if checkpoint.exists() and any(checkpoint.iterdir()):\n        raise FileExistsError(f\'Checkpoint directory is not empty: {checkpoint}\')\n\n    sys.path.insert(0, str(rmb_root.parent))\n    from robo_manip_baselines.common.base.TrainBase import TrainBase\n\n    def fixed_files(self):\n        self.all_filenames = ordered_files.copy()\n        print(\'Experiment 006-B fixed split: train=80 validation=20\', flush=True)\n        print(\'First training:\', Path(ordered_files[0]).name, flush=True)\n        print(\'First validation:\', Path(ordered_files[80]).name, flush=True)\n\n    original_stats = TrainBase.set_data_stats\n    def train_only_stats(self):\n        original_files = self.all_filenames\n        try:\n            self.all_filenames = original_files[:80]\n            print(\'Normalization: training 80 episodes ONLY\', flush=True)\n            return original_stats(self)\n        finally:\n            self.all_filenames = original_files\n\n    TrainBase.setup_rmb_files = fixed_files\n    TrainBase.set_data_stats = train_only_stats\n    sys.argv = [\n        str(entry), \'Act\',\n        \'--dataset_dir\', str(dataset),\n        \'--checkpoint_dir\', str(checkpoint),\n        \'--state_keys\', \'measured_joint_pos\',\n        \'--action_keys\', \'command_joint_pos\',\n        \'--camera_names\', \'front\',\n        \'--train_ratio\', \'0.8\',\n        \'--num_epochs\', str(args.epochs),\n        \'--batch_size\', str(args.batch_size),\n        \'--chunk_size\', str(args.chunk_size),\n        \'--skip\', str(args.skip),\n        \'--num_workers\', str(args.num_workers),\n        \'--seed\', \'42\',\n        \'--no-enable_rmb_cache\',\n    ]\n    print(\'Fresh training: no pretrained ACT checkpoint\', flush=True)\n    runpy.run_path(str(entry), run_name=\'__main__\')\n\nif __name__ == \'__main__\':\n    main()\n', encoding='utf-8')
py_compile.compile(str(TRAIN_SCRIPT), doraise=True)
print('Training script:', TRAIN_SCRIPT)
print('RMB:', RMB_REPO)


## 6. ACT新規学習 — 100 epochs

**このセルを実行すると学習が開始されます。** 途中までのチェックポイントが保存先に存在する場合は上書きを防ぐため実行しません。エラー調査に備え、全ログをDriveの学習用チェックポイントディレクトリではなく `/content/act_experiment_006b_train.log` に逐次記録します（学習後にDriveへコピー）。

In [ ]:
import os
import subprocess
from pathlib import Path

RMB_ROOT = RMB_REPO / 'robo_manip_baselines'
assert (RMB_ROOT / 'bin/Train.py').is_file()
assert TRAIN_SCRIPT.is_file()
assert not (CHECKPOINT_DIR.exists() and any(CHECKPOINT_DIR.iterdir())), f'出力先が空ではありません: {CHECKPOINT_DIR}'

EPOCHS = 100
BATCH_SIZE = 4
CHUNK_SIZE = 25
SKIP = 2
NUM_WORKERS = 2

env = os.environ.copy()
command = [
    str(PYTHON312), str(TRAIN_SCRIPT),
    '--dataset-dir', str(DATASET_DIR),
    '--split-json', str(SPLIT_JSON),
    '--checkpoint-dir', str(CHECKPOINT_DIR),
    '--rmb-root', str(RMB_ROOT),
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--chunk-size', str(CHUNK_SIZE),
    '--skip', str(SKIP),
    '--num-workers', str(NUM_WORKERS),
]
print('Running:', ' '.join(command), flush=True)
log_path = Path('/content/act_experiment_006b_train.log')
with log_path.open('w', encoding='utf-8') as log:
    with subprocess.Popen(command, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1) as proc:
        for line in proc.stdout:
            print(line, end='', flush=True)
            log.write(line)
            log.flush()
        returncode = proc.wait()
print('Exit code:', returncode)
if returncode != 0:
    print('=== Error log (last 80 lines) ===')
    print('\n'.join(log_path.read_text(errors='replace').splitlines()[-80:]))
    raise RuntimeError(f'ACT training failed. Full log: {log_path}')


## 7. チェックポイント確認と学習ログ保存

In [ ]:
import shutil

for name in ['model_meta_info.pkl', 'policy_best.ckpt', 'policy_last.ckpt']:
    path = CHECKPOINT_DIR / name
    print(name, 'OK' if path.is_file() else 'MISSING',
          f'{path.stat().st_size / 1e6:.2f} MB' if path.is_file() else '')
    assert path.is_file(), path

if log_path.is_file():
    dest = CHECKPOINT_DIR / 'train_v006b.log'
    shutil.copy2(log_path, dest)
    print('Saved log:', dest)
print('Experiment 006-B checkpoint saved. ACT単独成功はMuJoCo評価で確認してください。')
